# Windtunnel — how the whole system works

*Test the design under load before you build the aircraft.*

<img src="frontend/src/img/ReadyOrNotColour.png" alt="Innovation Month 2026 — Ready or Not…" width="300">

> **An Innovation Month 2026 (IM2026) entry.** Windtunnel is a demonstration system built for
> Innovation Month 2026 — *Ready or Not…*

This document is a **complete, plain-language tour** of Windtunnel: what it does, what each part is,
and how the pieces fit together. It is written for a curious reader — a public servant, a reviewer,
a future maintainer — who wants to genuinely understand the system without wading through code or
architecture jargon. Wherever it helps, it shows you the **real interface**, the **real agents**,
and the **real output**, using one worked example the whole way through: an *ATO Deduction Assistant*
chatbot, taken from an actual run.

If you want the authoritative detail behind any part, the closing section points you to the document
that owns it (`PROJECT_BRIEF.md` for intent, `TECH_SPEC.md` for the build, `DESIGN_BRIEF.md` for the
interface).

## In one paragraph

Windtunnel takes a public servant's loose idea for an AI solution and carries it through two phases.
The **Brainstorm phase** is a co-design conversation that sharpens the idea into a structured
outline (optionally with a clickable proof of concept and an information-flow map). The
**Governance phase** then stress-tests that outline with a college of specialist AI agents, assessing
the concept against the Australian Government's *AI impact assessment tool* (the DTA instrument, v1.0)
and producing a substantially complete **draft impact assessment** — as a notebook and a shareable
report. It compresses the distance between *"I have an idea"* and *"here is a well-argued draft
assessment my experts can review"* from weeks to hours. The metaphor is the point: **test the design
under load before you build the aircraft.**

> ### The one rule that governs everything below: *models argue, code computes.*
>
> AI agents may reason about how bad a risk would be and how likely it is — but **they never assign a
> risk rating.** Every rating is worked out by plain, fixed code from the government tool's own risk
> matrix. This single rule removes the most attackable weakness a system like this could have: it means
> no rating can ever be *talked* into a softer result, and every number is provably an output of code,
> not an opinion. You will see this rule show up again and again.

> ### Two honest limits, stated up front
>
> - **This is a draft, never an approval.** What Windtunnel produces is a draft for subject-matter
>   experts and an assessing officer to review. It is not legal advice, does not authorise any AI use,
>   and does not replace the human roles the AI policy defines.
> - **Everything here is public.** Every submission and every generated artefact is saved to a
>   world-readable repository. That is a deliberate design choice (explained below), disclosed to every
>   user before they type anything — the rule is simply: don't enter anything sensitive, classified, or
>   personal.

# Part I · The big picture

Three short ideas explain the whole shape of Windtunnel: **the two phases**, **the three programs and
one shared store**, and **the life of a single run**. Everything in Parts II–IV is detail hung on this
frame.

## 1. The journey, end to end

Follow one idea — *"a chatbot, hosted by the ATO, that helps the public understand what they can
deduct"* — all the way through.

1. **You describe your idea in plain words.** A warm, curious interviewer talks it over with you,
   drafting a structured **outline** as you go and gently pushing on the parts you hadn't thought about.
   You can also ask it to generate a **clickable mock-up** of the idea and a **map of how information
   would flow** through it.

2. **You submit the outline for assessment.** From here on it runs itself, and you can watch.

3. **A quick "threshold" check happens first.** Two independent assessors read your idea and judge how
   risky it is; a third agent reconciles them; and then *code* — not an AI — computes the risk ratings.
   If everything comes back low-risk, you can stop here. If anything is medium or high, a full
   assessment is warranted.

4. **A college of six specialists does the deep assessment.** Security, privacy, ethics, legal, data
   governance and solution architecture — each reads its own shelf of official government sources and
   writes only the parts of the assessment it owns, citing every source it leans on. If a specialist
   genuinely needs a fact it can't find, it asks you — once, in a single batch.

5. **An architect and a reviewer finish it.** The architect writes an implementation plan that answers
   the risks the specialists raised; a reviewer audits the whole thing for gaps and contradictions.

6. **You get a draft assessment** — a notebook (the record) and a clean report (the thing you hand to
   your experts), both fully cited, both downloadable, both saved under a run code you can return to
   any time.

The rest of this document walks each of these in turn, with pictures.

## 2. The shape of the system — three programs, one shared store

Windtunnel is made of three running programs and one thing they all share. The important design idea
is that **nothing keeps important information in its own memory.** The three programs never talk to
each other directly — they coordinate entirely through **one public repository**, which is both the
durable store and a public audit trail.

![Architecture: the interface, the Brainstorm backend, and the Governance pipeline, all reading from and writing to one public GitHub repository.](docs/img/overview/01-architecture.png)

<sub>*The three programs and the shared store they coordinate through. The browser holds no keys; the always-on server is the only piece that never sleeps; the assessment workers are created on demand and thrown away after each run.*</sub>

**Why one public repository holds everything.** Making the repository public buys three things at once:
a genuine, inspectable audit trail (anyone can see exactly what happened in any run), unlimited free
computing time for the assessment, and a single place where truth lives. The accepted cost — everything
a user submits is world-readable — is disclosed to every user before they begin.

**Why the repository is the *only* store.** The always-on server can fall asleep when idle, and each
assessment worker is a brand-new, empty machine. So the design treats the repository — not any
program's memory — as the system of record. Any program can die and restart, and the run simply picks
up where it left off, because the truth is safely in version control. This is the source of
Windtunnel's resilience: **a run is never really "in flight" in a way that can be lost.**

## 3. The life of a run — a fixed path with two pauses

Every run is a folder named by its **run code** (for example `WT-7K3D-Q2`). The code *is* the address
of the run, so resuming is just looking up a folder, and the whole trail stays easy to navigate and
read aloud. The code is a locator, not a password — the repository is public by design.

A run always walks the same sequence of stages. After each stage it **saves a checkpoint**, then moves
on. Just **twice** does it deliberately stop and wait for a person; everywhere else it runs on its own.

![Run lifecycle: Brainstorm → Threshold (two assessors, reconcile, rate, your review) → Full (six specialists, your questions, architect, reviewer, assemble) → report.](docs/img/overview/02-run-lifecycle.png)

<sub>*The fixed path a run walks. The two amber markers are the only moments it waits for you — and the second (your questions) is skipped entirely if no specialist needed to ask anything. The dark steps run no AI at all.*</sub>

**Why it pauses by stopping completely.** The assessment runs on shared automation that must never sit
idle waiting for a human. So at each of the two pause points, the run writes its checkpoint, marks
itself *waiting for you*, and shuts down cleanly. When you act, a fresh worker starts and continues
from the saved state. A neat consequence: **a paused run and a crashed run resume by exactly the same
mechanism** — a fresh start that reads the last checkpoint and carries on. Paste your run code weeks
later and you land on the exact right screen.

# Part II · Phase 1 — Brainstorm

Brainstorm is the co-design conversation that turns a loose idea into a sharp, structured outline.
It runs live on the always-on server, because you are sitting there waiting for each reply.

## 4. First contact — an honest front door

Before you can type anything, one screen stands in the way. It is deliberately calm — no red borders,
no legalese — but it makes three things unmissable: **this is public**, **nothing sensitive**, and
**what you get back is a draft, not an approval.**

![The 'A few things to know first' gate: This is public / Nothing sensitive / Draft only.](docs/img/overview/03-usage-gate.png)

<sub>*The usage gate everyone passes once. It leads with the plainly verifiable fact — everything is public — and asks the user to make their own judgement about what to put in.*</sub>

Past the gate, the landing screen offers exactly one clear path in — **Start a new idea** — and a quiet
way back — **Resume a run** — for anyone returning with a code.

![Windtunnel landing page: 'Test your AI idea before you build it.'](docs/img/overview/04-landing.png)

<sub>*The landing screen. One primary action, one quiet secondary one, and the Innovation Month 2026 framing.*</sub>

## 5. The co-design conversation and the outline

This is the heart of Brainstorm. On the left you talk to an interviewer; on the right, an **outline of
your idea builds live**, section by section, as the conversation goes. The outline has nine fixed parts
— the problem, the solution, who's affected, the data, a walk-through of one ordinary use, the
alternatives, the interface, the constraints, and how you'd know it worked. When all nine are filled,
the outline is ready to test.

![Annotated Brainstorm: conversation on the left, the live nine-part outline canvas on the right.](docs/img/overview/05-brainstorm.png)

<sub>*The real Brainstorm screen for the ATO Deduction Assistant example. The outline on the right is the single source of truth for the concept — everything downstream is generated from it, never patched separately.*</sub>

The **outline is the single source of truth for the concept.** Every artefact downstream — the whole
assessment — is generated from it. That is why the interviewer works so hard to get it right, and why
you can edit any section by hand at any time.

## 6. The agents behind Brainstorm

Five small agents take turns during Brainstorm. Each is given a plain-language job. Here is what each
one does — and, so you can see exactly how they are steered, an excerpt of the **actual instructions**
the main one is given.

| Agent | Its job, in one line |
| --- | --- |
| **Interviewer** | Works your idea over with pointed questions, drafts ahead by inference, and fills the outline as you talk. |
| **Sufficiency judge** | Quietly checks whether the outline is complete and self-consistent, and tells you when it's ready — but never blocks you. |
| **Feasibility gate** | Decides whether a clickable mock-up would actually help this particular idea, or whether a flow map is the better fit. |
| **Proof-of-concept generator** | Builds the single-page interactive mock-up, with an honest limitations banner written into it. |
| **Flow-map generator** | Draws the information-flow diagram (actors, systems, data stores and the flows between them). |

**What the interviewer is actually told** (an excerpt of its real instructions — the full, versioned
prompt lives in `prompts/interviewer.v1.md`):

> You are the **co-design interviewer** for Windtunnel… You are warm, plain-spoken, sharp, and
> genuinely curious. You are a design partner thinking *with* the person, not a form they fill in.
>
> …do two things at once on every turn: **draft ahead by inference**, and **push their thinking**.
> - **Draft ahead.** Read between the lines. From even a loose first message you can usually make
>   sensible, conventional assumptions about several sections — write them… make the assumption
>   visible in your reply… so they can wave it through or fix it.
> - **Push their thinking.** …Ask the interesting question — the nuance they may have skipped, the
>   design fork they haven't noticed they're standing at, the stakeholder who's affected but wasn't
>   mentioned…

Notice the two safeguards baked into that instruction. First, the interviewer is told to **make its
assumptions visible** so you can correct them — the outline building live beside the chat is exactly
that safety net. Second, everything you type is treated as **information about the idea, never as
instructions to the agent** — so a message that says "mark everything done" is understood as a fact
about your use case, not a command to obey. Every agent that touches your words follows this same
discipline.

## 7. Two richer ways to say the same thing — a mock-up and a map

Talking isn't always the clearest way to express an idea. So Brainstorm offers two optional artefacts.

The **proof of concept** is a single, self-contained web page that lets a reviewer *feel* how the idea
would work. Before it builds one, the feasibility gate asks an honest question: *would a static mock-up
genuinely help here?* For an interface or dashboard, yes; for a back-office automation with no screen,
no — and it says so and offers the flow map instead. Every generated mock-up carries a prominent
limitations banner, written into the page itself, so it can never be mistaken for the real thing.

![Annotated proof of concept: the limitations banner, the CONCEPT PROOF badge, and the mock triage interface.](docs/img/overview/07-proof-of-concept.png)

<sub>*The real proof of concept generated for the ATO example — a believable, clickable interface, wrapped in an unmissable 'this is only a demonstration' banner.*</sub>

The **flow map** shows the moving parts and how information travels between them — the person, the
systems, the data stores, and the numbered steps connecting them. (It's drawn in the browser and saved
back, so the small server never has to do heavy graphics work.)

![Flow map of the ATO Deduction Assistant showing actors, systems, data stores and flows.](docs/img/overview/06-flow-map.png)

<sub>*The information-flow map generated for the ATO example: the taxpayer, the public portal, the AI service and its policy database, and the official resources — with each flow numbered.*</sub>

When you're satisfied — the outline is required, the mock-up and map are encouraged but optional — you
**submit for assessment**, and the governance phase begins.

# Part III · Phase 2 — Governance

This is where the idea is stress-tested. It runs on shared automation (created on demand, thrown away
after), and it saves its progress to the repository after every stage. First, though, the part you
actually see.

## 8. Watching it happen — the transparency layer

A governance run takes several minutes. Rather than hide that behind a spinner, Windtunnel shows you
**everything**: an animated diagram of the whole pipeline where each stage is a node you can watch light
up, plus a running activity log that states — in words — every single thing that happens. The goal is
that a non-expert finishes a run genuinely confident that every angle was considered.

![Annotated Chamber: the threshold band, the specialist college, the knowledge bases, live retrieval captions, the waiting downstream stages, the activity log, and the zoomable canvas.](docs/img/overview/08-chamber.png)

<sub>*The Chamber, mid-run. This is a real assessment paused in the middle of the specialist stage: the threshold band is finished, three specialists are actively reading their sources, and the later stages wait their turn.*</sub>

Two design rules keep this honest rather than decorative. First, **nothing exists only as a picture** —
every change of a node's state is also written out in words in the log, which is both the accessibility
backbone and a promise that the animation isn't theatre. Second, state is always shown by **label,
shape and position — never colour alone**, so it's legible to everyone.

Under the hood, the mechanism is simple and robust: the pipeline writes a fresh **status snapshot** to
the repository every few seconds, and the page asks the server "what's happening now?" on a loop, then
animates whatever is new. One snapshot fully describes the whole picture, so even if the page misses an
update — you close the laptop, you lose signal — it always redraws correctly from the next one.

## 9. The threshold assessment — is this even risky?

Governance opens with a quick question: *is this a low-risk idea that can stop here, or does it warrant
the full treatment?* The design here is deliberate and worth understanding.

**Two generalist assessors read your idea independently and in parallel.** They don't see each other's
work. This is on purpose: **disagreement between two independent assessors is a signal**, not noise —
and the government tool's own guidance says to resolve rating disagreements by taking the more cautious
view. Each assessor is explicitly **precautionary**: when the evidence is thin, it leans to the more
cautious judgement.

**What a threshold assessor is actually told** (excerpt; full prompt in
`prompts/threshold_generalist.v1.md`):

> You are an experienced Australian Government AI-impact assessing officer… explicitly
> **precautionary**: when the evidence is thin or ambiguous you lean toward the more cautious judgement
> rather than the more comfortable one.
>
> - **You never assert a risk rating.** You choose a consequence tier and a likelihood tier and explain
>   them. The risk rating… [is] computed by code from the tool's Table 2 — not by you.
> - **Precautionary defaults**… where you are uncertain… take the **higher** consequence tier you can
>   reasonably justify…

**A reconciler then merges the two drafts.** Where the assessors diverge on how bad something would be
or how likely it is, the reconciler **takes the higher of the two** and writes a short, visible note
explaining the disagreement. Crucially, the reconciler settles the *wording*, never the rating.

You then see the finished threshold assessment and decide what happens next.

![Threshold review screen: the inherent-risk table, the overall rating, and the assessor divergence notes.](docs/img/overview/09-threshold-review.png)

<sub>*The real threshold review for a 'Business Navigator' example. Every risk area has a consequence and a likelihood; below the table, each rationale is spelled out — and where the two assessors disagreed, a plain 'Divergence' note shows what each thought and why the more cautious view stands. Disagreement is surfaced, never hidden.*</sub>

Routing follows the tool's own logic, and **you are in control**:

- **All risks low** → you *may* conclude here; the artefact is framed as ready for an approving officer
  to consider.
- **Any risk medium or high** → a full assessment is warranted.
- Either way, you can choose the full assessment if you want it.

## 10. The rating engine — where *"models argue, code computes"* lives

This is the single most important integrity mechanism, so it's worth seeing plainly. For each risk
area, an agent supplies **only two things**: a **consequence** (how bad it would be) and a
**likelihood** (how probable it is), each with an evidenced rationale. A small piece of **fixed code**
then looks up the rating in the government tool's own risk matrix. No agent ever writes a rating.

![Annotated risk table: consequence and likelihood are argued by the AI; the rating is computed by code.](docs/img/overview/10-rating-engine.png)

<sub>*The residual-risk table from a real report. The AI argues only the consequence and the likelihood; the rating chip is computed by code from those two, the same way every time.*</sub>

A few things make this trustworthy:

- **The matrix is transcribed verbatim from the real government tool**, and lives in exactly one place.
  (Fidelity note: the real matrix is *not* the textbook 5×5 — it tops out at "High", with several cells
  that differ from the generic version. It was copied cell by cell, and the tests that check it are
  worked by hand from the actual tool. This is the most-tested part of the whole system, on purpose.)
- **It holds under revision.** If you later ask for a change, your request can reshape the *reasoning*
  that leads to a consequence or likelihood — but it can never set a rating. The engine always
  recomputes. So the number is *provably* an output of code from the resolved inputs.
- **The same engine computes the residual rating** (the risk that remains after the specialists'
  mitigations are in place), later in the run.

## 11. The specialist college — six experts, each in its lane

If a full assessment is warranted, six specialist agents take over. Each **owns a fixed set of the
assessment's sections and nothing else**, and each has its **own private library** of official
government sources. A specialist literally cannot edit another's work — not as a matter of politeness,
but structurally: each writes only its own sections, and anything out of scope is rejected before it
can enter the assessment.

| Specialist | Owns (assessment sections) | Its own library | What's on that shelf |
| --- | --- | --- | --- |
| **IT Security** | 6.7, 7.3 | 13 documents | Information Security Manual, the Essential Eight, cloud-controls and AI-security guidance |
| **Privacy** | 7.1, 7.2 | 8 documents | the *Privacy Act 1988*, the Australian Privacy Principles, OAIC guidance |
| **Ethics & Fairness** | 5.1, 5.2, 8.1, 8.2, 8.4, 8.5, 10.1 | 17 documents | Australia's AI Ethics Principles, assurance frameworks, human-rights guidance |
| **Legal & Administrative Law** | 9.1, 9.2, 10.2, 11.1, 12.1, 12.2 | 11 documents | administrative-law & automated-decision guidance, the whole-of-government AI policy |
| **Data Governance** | 6.1, 6.2, 8.3 | 5 documents | the *Archives Act*, records authorities, data-quality and data-strategy guidance |
| **Solution Architect** | 6.3–6.6, 6.8 (+ the implementation plan) | 56 documents | hosting-certification, approved-pattern, AI-suitability and technical-standard material |

Every specialist works from: your outline, the finished threshold assessment, its own library, and the
government tool's exact question text for its owned sections. It has broad freedom in *how* it writes —
but one hard rule: **every claim that leans on a source must cite that source and a precise, checkable
pointer** (a real page for a PDF; a provision or heading otherwise). Where its material genuinely
doesn't cover something, it does **not** guess — it records the gap and recommends a concrete next step.

**What a specialist is actually told** (excerpt; full prompt in `prompts/specialist.v1.md`):

> You are a subject-matter specialist… You have been given **one narrow slice of the tool** — a fixed
> set of owned sections — and a private knowledge base of your own reference material. You work
> carefully, cite only what you actually read, and never touch anything outside your slice.
>
> For **every** owned section, do exactly one of:
> - **Draft it** — a grounded, evidenced answer… cite fetched material as `(short_name, locator)`…
> - **Flag it as a gap**… when the outline and your library genuinely do not give you enough to answer.
>   A gap is honest; a fabricated answer is not.

## 12. How a specialist reads its library — and why it's unusual

Most AI systems that "read documents" work like a search box: you guess a query, it fetches whatever
looks similar, and the model writes from that. Windtunnel deliberately does **not** do this, for one
decisive reason: with a search box, a **miss is silent** — the model never knows the one crucial
document it needed was left out. For a governance assessment, silent gaps are the one failure it cannot
afford.

So instead, each specialist is handed **a catalogue of its entire library** — a map listing every
document, its structure, and what each part contains — and **chooses what to read.** It can:

- **fetch** a specific thing by its address (a numbered security control, a named privacy principle, a
  particular section), or
- **search** its library by keyword as a backstop.

Because the whole map is in front of it, a gap is **visible and recoverable** — the specialist can see
it missed something and go back for it. It reads over a few short rounds, then writes. And it can only
ever cite what it actually opened, so **what it chose to read is itself part of the audit trail** (it's
what you saw scrolling past in the activity log).

The libraries themselves are built ahead of time by splitting each source **along its own structure** —
by section, by provision, by numbered control — so that every quoted passage carries an exact,
human-checkable pointer back to where it came from. And because the repository is public, a strict rule
guards the door: **a document can only enter a library if it is cleared as freely redistributable.** No
clearance, no entry.

Here is what that grounding looks like in the finished report — inline pointers on every claim, and a
tidy "Sources" line under each section:

![Report section 6.1 with inline pinpoint citations and a Sources list.](docs/img/overview/12-citations.png)

<sub>*A real specialist section (data governance) in the finished report. Every substantive claim carries a precise citation like (Data Quality Checklist, p.1), and the section ends with a Sources line. Where a section doesn't apply, it says so plainly rather than inventing content.*</sub>

## 13. The question checkpoint — the one place it asks you

Sometimes a section turns on a fact your outline simply doesn't state and no library can supply — *where
will the privacy assessment be stored? is there a pilot planned?* When that happens, a specialist may
ask you. The rules are tight: **at most three questions each, only when guessing would be genuinely
inappropriate**, and the happy path is **zero** questions.

Mechanically it's a single pause: all specialists draft, every question is gathered into one batch, the
run stops once, and you answer what you can. **Anything you skip is written up honestly as a gap** — it
is never quietly guessed.

![The 'specialists have a few questions' checkpoint, grouped by specialist with skip options.](docs/img/overview/11-question-checkpoint.png)

<sub>*The question checkpoint from a real run. Questions are grouped by the specialist that asked, each with a one-line reason. Every question can be answered or skipped — and skipping is explicitly fine.*</sub>

## 14. The architect and the reviewer — finishing and checking

Once every specialist is done, two senior agents close the assessment out.

**The solution architect** puts on a second hat. Having already drafted its own technical sections, it
now reads the *whole* completed assessment and writes an **implementation plan** — how you'd actually
build, host and run this — where **every step traces back to a specific mitigation a specialist called
for.** The plan exists to *demonstrably answer* the assessment, not sit beside it.

> You are the **solution architect**… write a concrete plan for **how to actually build and deploy this
> system in a way that answers the risks and implements the mitigations the specialists identified**…
> every step you write must trace back to a specific mitigation, control, or requirement a specialist
> actually recorded. *(excerpt; `prompts/architect.v1.md`)*

**The adjudicating reviewer** then audits the assembled draft for two things: **coverage** (is every
question either answered or honestly logged as a gap?) and **coherence** (does anything contradict
anything else?). When it finds a genuine conflict, it doesn't rewrite — it rules on which side is
better supported and directs *that* specialist to fix its own section. But its most interesting
instruction is what it does when both sides are genuinely sound:

> **Unresolved disagreements.** Where both sides are genuinely well supported and the conflict turns on
> a human policy judgement, do **not** force a resolution… honest disagreement is more credible than
> manufactured agreement. *(excerpt; `prompts/reviewer.v1.md`)*

So the assessment can end with a clearly-labelled *"points of unresolved disagreement"* — two
well-argued positions the system deliberately declined to collapse into false consensus. For a
governance document, that honesty is the point. The reviewer's correction loop is capped at two rounds,
and it also computes (via the same rating engine) the **residual risk** that remains after mitigation.

# Part IV · The result, and the fine print

## 15. What you walk away with

The run ends with two artefacts, both saved under your run code and both downloadable: a **notebook**
(the complete record, assembled programmatically so nothing "runs" or can drift) and a clean **report**
(the shareable version your experts actually read). The report mirrors the government tool's structure
exactly — the threshold sections with their computed rating chips and divergence notes, then the full
sections in order with every citation resolved, the residual-risk summary, the implementation plan,
and the appendices.

![The report view: 'Your draft assessment is ready', with open/download actions and the report itself.](docs/img/overview/13-report.png)

<sub>*The finished report, open in the app. It is clearly marked DRAFT — FOR SME REVIEW, carries the run code and generation time, and sits beside an honest 'ask for a revision' option (capped at two rounds).*</sub>

Two appendices deserve a special mention, because they are where Windtunnel's honesty is most visible.

The **recommended next steps** register gathers every gap the assessment left open — including any
questions you skipped and, notably, the one task **no AI is allowed to do**: the internal governance
body's own review, which is emitted as a flagged human action rather than faked.

![The 'recommended next steps' / gap register, listing unanswered questions as honest gaps.](docs/img/overview/14-gap-register.png)

<sub>*The gap register in a real report. Skipped checkpoint questions and things the assessment couldn't safely determine are listed openly as work still to do — not papered over.*</sub>

The **provenance** appendix records everything needed to reproduce and trust the result. From a real
run:

- **Run id and timestamps** — when it was created and generated.
- **Which model and prompt version played each role**, for example:
  - *specialist* and *threshold assessor* → the fast drafting model (`gemini-3.5-flash`)
  - *reconciler, architect, reviewer* → the deep-reasoning model (`gemini-3.1-pro-preview`)
- **Which library version each specialist read**, with document counts (13, 8, 17, 11, 5, 56).
- **The submitting officer's attestation** — a recorded confirmation that the inputs contained nothing
  sensitive, classified or personal (appropriate for a public repository; it is not a claim that the
  system is accredited to any sensitivity level).
- **A full, de-duplicated reference list** — the pinpoint-cited apparatus behind every claim.

## 16. What a curious person might still ask

**Which AI models does it use?** Three tiers, each chosen to fit its job (and all named in every
report's provenance). A **lite** model runs the chatty parts of Brainstorm — the interviewer that talks
with you and drafts your outline, plus the quick sufficiency and feasibility checks. A **fast** model
does the heavier drafting — the proof-of-concept and flow map, the two threshold assessors, and the six
specialists. A **deep-reasoning** model handles the jobs that turn on real judgement — the reconciler,
the architect, and the reviewer. No model ever computes a rating; that is always code.

**Where do the official sources come from, and is that legal to republish?** The libraries are built
from roughly 110 government and standards documents. Because the repository is public, each quoted
passage is effectively republished — so the build **refuses any document not cleared as freely
redistributable.** It's a hard gate, not a preference.

**How safe is it with what I type?** Treat it as fully public. It carries no security accreditation and
makes no claim about what sensitivity it can handle. Every agent that reads your words treats them as
information about your idea, never as instructions — but the real protection is the rule stated at the
front door: don't put in anything you wouldn't be comfortable seeing in the open.

**Can I change my mind?** Yes, within honest limits. The Brainstorm conversation is unbounded. After
they're first made, the mock-up, the flow map, the threshold assessment and the full assessment each
allow **up to two** rounds of your revisions. The reviewer's own internal correction loop is separately
capped at two.

**What if something breaks mid-run?** It becomes a calm, resumable state, not a crash: a plain message
for you, the technical detail tucked out of the way, and the last good checkpoint intact. Resume with
your run code and it continues.

**What can't it do?** It cannot approve anything, it cannot give legal advice, and it explicitly will
not perform the internal governance body's review — that is a human responsibility, and the assessment
flags it as one rather than pretending otherwise.

---

## Where to go deeper

This document is a map, not the territory. The authoritative detail lives in the foundational documents,
which win over this overview wherever they and it ever disagree.

| You want the detail on… | Read |
| --- | --- |
| *Why* the product is shaped this way; the decisions taken as fixed | `PROJECT_BRIEF.md` |
| The pipeline, the data contracts, the libraries, the state machine, the rating engine | `TECH_SPEC.md` |
| The interface, the transparency animation, and the report styling | `DESIGN_BRIEF.md` |
| The government instrument itself (the risk tables and consequence descriptors) | `instrument/guidance/` |
| The exact, versioned instructions each agent is given | `prompts/` |
| What's built, what's in flight, and the decisions taken along the way | `STATUS.md` |
| The conventions and invariants for anyone editing the repo | `CLAUDE.md` |

*The project brief governs intent. The design brief governs the interface and the report. The tech spec
governs the pipeline, the data contracts, and the build. This overview simply tries to make the whole
thing legible.*

<sub>Screenshots in this document are from real runs (the *ATO Deduction Assistant* and *Business
Navigator* examples); the run state is served from the repository's own `runs/` history.</sub>